In [64]:
from datasets import load_dataset
import torch
import numpy as np
import chess
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from datasets import concatenate_datasets

In [65]:
dataset = load_dataset(
    "mateuszgrzyb/lichess-stockfish-normalized",
    split="train",
    cache_dir="./training_data"
)

Loading dataset shards:   0%|          | 0/35 [00:00<?, ?it/s]

In [66]:
min_depth = 30

filtered_ds = dataset.filter(lambda x: x["depth"] >= min_depth, num_proc=8)
print(len(filtered_ds))

split1 = filtered_ds.train_test_split(test_size=0.2, seed=42)

train_ds = split1["train"]
temp_ds  = split1["test"]

# second split: val vs test
split2 = temp_ds.train_test_split(test_size=0.5, seed=42)

val_ds   = split2["train"]
test_ds  = split2["test"]

66271195


In [67]:
#see https://official-stockfish.github.io/docs/nnue-pytorch-wiki/docs/nnue.html#halfkp
PIECE_MAP = {
    chess.PAWN: 0,
    chess.KNIGHT: 1,
    chess.BISHOP: 2,
    chess.ROOK: 3,
    chess.QUEEN: 4,
}

# using the same formula provided by the stockfish site
def halfkp_index(piece, piece_square, king_square, perspective): # perspective is who is "us"
    piece_type = PIECE_MAP[piece.piece_type]
    piece_color = 0 if piece.color == perspective else 1
    p_idx = piece_type * 2 + piece_color
    return piece_square + (p_idx + king_square * 10) * 64

def flip_square(sq):
    file = sq % 8
    rank = sq // 8
    return (7 - rank) * 8 + file

def extract_halfkp(board, perspective):
    feats = []

    king_sq = board.king(perspective)

    if perspective == chess.BLACK:
        king_sq = flip_square(king_sq)

    for sq, piece in board.piece_map().items():
        if piece.piece_type == chess.KING:
            continue

        if perspective == chess.BLACK:
            sq = flip_square(sq)

        feats.append(
            halfkp_index(piece, sq, king_sq, perspective)
        )

    return feats

In [ ]:
class ChessDataset(Dataset):
    def __init__(self, data, max_samples=None):
        ds = data if max_samples is None else data.select(range(max_samples))

        self.non_zero = ds.filter(lambda x: x["cp"] is not None and abs(x["cp"]) > 20)
        self.zero = ds.filter(lambda x: x["cp"] is not None and abs(x["cp"]) <= 20)

        # enforce 80/20 mix
        n_non_zero = int(0.8 * len(ds))
        n_zero = int(0.2 * len(ds))

        non_zero_part = self.non_zero.shuffle(seed=42).select(
            range(min(n_non_zero, len(self.non_zero)))
        )

        zero_part = self.zero.shuffle(seed=42).select(
            range(min(n_zero, len(self.zero)))
        )

        self.ds = concatenate_datasets([non_zero_part, zero_part]).shuffle(seed=42)

        self.ds = self.ds.shuffle(seed=42)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]

        board = chess.Board(item["fen"])

        white_feats = extract_halfkp(board, chess.WHITE)
        black_feats = extract_halfkp(board, chess.BLACK)

        stm = 1.0 if board.turn == chess.WHITE else 0.0

        SCALE = 400.0
        cp = 0.0 if item["cp"] is None else item["cp"]

        #wdl = torch.sigmoid(torch.tensor(cp / SCALE, dtype=torch.float32))

        cp = np.clip(cp, -10000, 10000)
        cp = cp / 1000.0
        #cp = np.tanh(cp / 800.0) * 800.0 

        return {
            "white": torch.tensor(white_feats, dtype=torch.long),
            "black": torch.tensor(black_feats, dtype=torch.long),
            "stm": torch.tensor(stm, dtype=torch.float32),
            "target": torch.tensor([cp], dtype=torch.float32)
        }

In [69]:
def collate_fn(batch):
    B = len(batch)

    white_out = torch.zeros(B, NUM_FEATURES, dtype=torch.float32)
    black_out = torch.zeros(B, NUM_FEATURES, dtype=torch.float32)

    for i, b in enumerate(batch):
        white_out[i, b["white"]] = 1.0
        black_out[i, b["black"]] = 1.0

    stm = torch.tensor([b["stm"] for b in batch], dtype=torch.float32).unsqueeze(1)
    target = torch.stack([b["target"] for b in batch])

    return white_out, black_out, stm, target

In [70]:
NUM_FEATURES=40960
M = 256
N = 32
K = 1

class NNUE(nn.Module):
    def __init__(self):
        super().__init__()

        self.ft = nn.Linear(NUM_FEATURES, M)
        self.l1 = nn.Linear(2 * M, N)
        self.l2 = nn.Linear(N, K)

    # The inputs are a whole batch!
    # `stm` indicates whether white is the side to move. 1 = true, 0 = false.
    def forward(self, white_features, black_features, stm):
        w = self.ft(white_features)
        b = self.ft(black_features)

        accumulator = (stm * torch.cat([w, b], dim=1)) + ((1 - stm) * torch.cat([b, w], dim=1))

        x = torch.clamp(accumulator, 0.0, 1.0)
        x = torch.clamp(self.l1(x), 0.0, 1.0)

        return self.l2(x)

In [71]:
train_data = ChessDataset(train_ds, max_samples=100000)
val_data   = ChessDataset(val_ds, max_samples=20000)
test_data  = ChessDataset(test_ds, max_samples=50000)

train_loader = DataLoader(
    train_data,
    batch_size=128,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_data,
    batch_size=128,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_data,
    batch_size=128,
    shuffle=False,
    collate_fn=collate_fn
)

In [72]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

In [73]:
model = NNUE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.MSELoss()

In [74]:
EPOCHS = 20

for epoch in range(EPOCHS):

    # --------------------
    # TRAIN
    # --------------------
    model.train()
    train_loss = 0.0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]")

    for white, black, stm, target in train_bar:

        white = white.to(device)
        black = black.to(device)
        stm = stm.to(device)
        target = target.to(device)

        pred = model(white, black, stm)
        loss = loss_fn(pred, target)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()
        train_bar.set_postfix(loss=loss.item())

    # --------------------
    # VALIDATION
    # --------------------
    model.eval()
    val_loss = 0.0

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1} [VAL]")

    with torch.no_grad():
        for white, black, stm, target in val_bar:

            white = white.to(device)
            black = black.to(device)
            stm = stm.to(device)
            target = target.to(device)

            pred = model(white, black, stm)
            loss = loss_fn(pred, target)

            val_loss += loss.item()
            val_bar.set_postfix(loss=loss.item())

    train_loss /= len(train_loader)
    val_loss /= len(val_loader)

    print(
        f"\nEpoch {epoch+1} Summary | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}\n"
    )

Epoch 1 [VAL]: 100%|██████████| 74/74 [00:02<00:00, 25.02it/s, loss=3.8e+4] 



Epoch 1 Summary | Train Loss: 64530.741904 | Val Loss: 65929.720914



Epoch 2 [VAL]: 100%|██████████| 74/74 [00:02<00:00, 30.33it/s, loss=3.8e+4] 



Epoch 2 Summary | Train Loss: 64444.715192 | Val Loss: 65886.526077



Epoch 3 [VAL]: 100%|██████████| 74/74 [00:02<00:00, 30.00it/s, loss=3.79e+4]


...


Epoch 18 Summary | Train Loss: 62570.833096 | Val Loss: 65821.667230



Epoch 19 [VAL]: 100%|██████████| 74/74 [00:02<00:00, 28.41it/s, loss=3.77e+4]



Epoch 19 Summary | Train Loss: 62472.320007 | Val Loss: 65810.455184



Epoch 20 [VAL]: 100%|██████████| 74/74 [00:02<00:00, 29.70it/s, loss=3.77e+4]


Epoch 20 Summary | Train Loss: 62377.787446 | Val Loss: 65817.035737



In [75]:
test_loss = 0.0
test_cp_mae = 0.0
test_sign_correct = 0
test_samples = 0

model.eval()

all_errors = []

test_bar = tqdm(test_loader, desc="[TEST]")

with torch.no_grad():
    for white, black, stm, target in test_bar:

        white = white.to(device)
        black = black.to(device)
        stm = stm.to(device)
        target = target.to(device)

        pred = model(white, black, stm)
        loss = loss_fn(pred, target)

        pred_cp = pred * 1000
        target_cp = target * 1000

        errors_cp = pred_cp - target_cp
        all_errors.append(errors_cp.view(-1).cpu())

        test_cp_mae += torch.abs(errors_cp).sum().item()

        sign_correct = ((pred > 0) == (target > 0)).sum().item()

        bs = target.size(0)

        test_loss += loss.item() * bs
        test_sign_correct += sign_correct
        test_samples += bs

        test_bar.set_postfix(loss=loss.item())

print("\nFINAL TEST RESULTS")
print(f"Test loss: {test_loss / test_samples:.6f}")
print(f"Test MAE (cp): {test_cp_mae / test_samples:.2f}")
print(f"Sign accuracy: {test_sign_correct / test_samples:.3f}")


errors = torch.cat(all_errors).numpy()

print("\nERROR STATS (cp)")
print("Mean:", errors.mean())
print("Std:", errors.std())
print("Median:", np.median(errors))
print("MAE:", np.mean(np.abs(errors)))
print("95th percentile:", np.percentile(errors, 95))
print("-95th percentile:", np.percentile(errors, 5))

[TEST]: 100%|██████████| 183/183 [00:08<00:00, 21.67it/s, loss=7.61e+4]


FINAL TEST RESULTS
Test loss: 64083.778779
Test MAE (cp): 141473.39
Sign accuracy: 0.493

ERROR STATS (cp)
Mean: -21024.781
Std: 252273.12
Median: 281.62854
MAE: 141473.39
95th percentile: 434688.1187499999
-95th percentile: -498479.4


In [76]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": epoch,
    "loss": train_loss
}, "nnue_orig.pt")

In [79]:
def evaluate_fen(model, fen, device):
    model.eval()

    board = chess.Board(fen)

    # extract HalfKP-style index lists
    white_feats = extract_halfkp(board, chess.WHITE)
    black_feats = extract_halfkp(board, chess.BLACK)

    # multi-hot encode (same as collate_fn logic)
    white_vec = torch.zeros(NUM_FEATURES, dtype=torch.float32)
    black_vec = torch.zeros(NUM_FEATURES, dtype=torch.float32)

    white_idx = torch.tensor(white_feats, dtype=torch.long)
    black_idx = torch.tensor(black_feats, dtype=torch.long)

    white_vec.scatter_(0, white_idx, 1.0)
    black_vec.scatter_(0, black_idx, 1.0)

    # batch dimension
    white_vec = white_vec.unsqueeze(0).to(device)
    black_vec = black_vec.unsqueeze(0).to(device)

    stm = torch.tensor([[1.0 if board.turn == chess.WHITE else 0.0]],
                        dtype=torch.float32).to(device)

    with torch.no_grad():
        pred = model(white_vec, black_vec, stm)
        
        
    return pred

In [80]:
fen = "r1bqk1nr/pppp1ppp/2n5/2b1p1N1/2B1P3/8/PPPP1PPP/RNBQK2R b KQkq - 0 1"
print(evaluate_fen(model, fen, device))

fen2 = "r1bNk2r/pppp2pp/2n4n/2b1p3/2B1P3/8/PPPP1PPP/RNBQK2R w KQkq - 0 1"
print(evaluate_fen(model, fen2, device))

fen3 = "rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR b KQkq e3 0 1"
print(evaluate_fen(model, fen3, device))

tensor([[10.4860]], device='mps:0')
tensor([[10.4860]], device='mps:0')
tensor([[10.4860]], device='mps:0')
